# Taxi Simulation — Batch Run (Google Colab)

This notebook clones the project from GitHub, installs dependencies, and does a batch run of the simulation.

**Usage:**
1. Run the *Setup* cell once to clone the repo and install dependencies.
2. Run the *Generate configs* cell to create new config files.
3. Run the *Move configs* cell to collect the generated files into a subfolder.
4. Edit `CONFIG_FOLDER` in the *Batch run* cell to point to that subfolder, then run it.
5. Download results.

## Setup — clone repo & install dependencies

In [ ]:
import os

REPO_URL = "https://github.com/Tmmn/taxi.git"
REPO_DIR = "taxi"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}
else:
    print("Repo already cloned — pulling latest changes...")
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

In [ ]:
# Install required Python packages
!pip install numpy scipy matplotlib

## Generate config files

Skip this cell if you already have the config files you want to run.

Available modes: `sweep`, `passenger_fairness`, `region_pref`, `distance_pref`, `passenger_pref`, `two_sided`, `safety_objective`, `simple`

See chapter [Config generation](https://github.com/Tmmn/taxi/blob/main/README.md#config-generation) in the README.md for the full list of arguments per mode.

In [ ]:
# Example: generate passenger_pref configs based on the big_city_base config for 5 simulated days
# Adjust the mode and arguments as needed.
!python generate_configs.py passenger_pref big_city_base.conf 5 12

## Move generated configs to a batch folder

Most modes write config files flat into `configs/`. Move them into a dedicated subfolder before running the batch, so `batch_run.py` can pick them up.

Set `GLOB_PATTERN` to match the files you just generated (based on the base config name and days used above) and `TARGET_FOLDER` to your desired subfolder.

In [ ]:
import glob, shutil, os

GLOB_PATTERN = "configs/big_city_base_days_*.conf"  # match the files you just generated
TARGET_FOLDER = "configs/passenger_pref"            # destination subfolder

os.makedirs(TARGET_FOLDER, exist_ok=True)
moved = 0
for f in glob.glob(GLOB_PATTERN):
    shutil.move(f, os.path.join(TARGET_FOLDER, os.path.basename(f)))
    moved += 1
print(f"Moved {moved} config(s) to {TARGET_FOLDER}/")

## Batch run

Set `CONFIG_FOLDER` to the directory that contains the `.conf` files you want to run.
Results are written to `results/<subfolder>/` and per-run logs to `batch_run_logs/`.

In [ ]:
CONFIG_FOLDER = "configs/passenger_fairness"  # <-- change this to your config folder

!python batch_run.py {CONFIG_FOLDER}

## Download results

Zip and download the results folder to your local machine.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("results", "zip", "results")
files.download("results.zip")